In [35]:
import pandas as pd
from openai import OpenAI
import nltk
from tqdm import tqdm
import json
import os

# Baixar o tokenizador de sentenças (necessário apenas na primeira vez)
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/ricardo/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
# Configuração do Cliente LM Studio (Compatível com AWS SageMaker LMI)
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# Identificadores conforme carregados no seu LM Studio
MODEL_GEN = "lmstudio-community/Qwen3.5-9B-GGUF"
MODEL_GUARD = "PatronusAI/Llama-3-Patronus-Lynx-8B-Instruct-Q4_K_M-GGUF"

In [41]:
def carregar_dataset_seguro(csv_path):
    try:
        # Tentativa 1: Padrão brasileiro (ponto-e-vírgula) com tratamento de erros
        df_full = pd.read_csv(
            csv_path, 
            sep=';', 
            encoding='utf-8', 
            on_bad_lines='skip', # Pula linhas malformadas para não travar o experimento
            engine='python'      # Engine Python é mais lenta mas mais flexível com erros de parsing
        )
        # Se o dataframe vier com apenas 1 coluna, provavelmente o separador era vírgula
        if df_full.shape[1] <= 1:
            df_full = pd.read_csv(csv_path, sep=',', encoding='utf-8', on_bad_lines='skip')
            
        print(f"Dataset carregado com sucesso: {len(df_full)} registros.")
        return df_full
    except Exception as e:
        print(f"Erro crítico ao carregar CSV: {e}")
        return None

In [56]:
# Caracteres de controle como \r ou \t podem afetar a tokenização e o desempenho da NLI.
def limpar_texto_policial(texto):
    if pd.isna(texto): return ""
    # Remove quebras de linha e excesso de espaços para manter a densidade factual
    texto_limpo = texto.replace('\n', ' ').replace('\r', ' ').strip()
    return " ".join(texto_limpo.split())

# Aplicação no loop do experimento
# natureza = limpar_texto_policial(row)
# fato_original = limpar_texto_policial(row)

In [60]:
def processar_experimento(csv_path, n_amostras=100):
    # 1. Carregar Dataset (ajuste os nomes das colunas conforme o seu CSV do SPSafe)
    df_full = carregar_dataset_seguro(csv_path)
    df = df_full.sample(n=min(n_amostras, len(df_full)), random_state=42)
    
    # Inicialização da lista corrigida
    resultados_finais = []

    print(f"Iniciando auditoria de {len(df)} registros policiais...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        contexto_referencia = f"Relato: {row['NARRATIVA']}"
        
        # 2. GERAÇÃO: O Qwen 3.5 redige o relatório formal
        try:
            prompt_gen = f"Com base no seguinte fato policial, escreva um relatório técnico resumido: {contexto_referencia}"
            gen_resp = client.chat.completions.create(
                model=MODEL_GEN,
                messages=[{"role": "user", "content": prompt_gen}],
                temperature=0.7
            )
            relatorio_ia = gen_resp.choices[0].message.content
            
            # 3. FRAGMENTAÇÃO: Quebra o relatório em sentenças atômicas
            sentencas = nltk.sent_tokenize(relatorio_ia)
            
            # 4. AUDITORIA: O Lynx-8B verifica cada sentença contra o B.O. original
            for sent in sentencas:
                # Prompt padrão do Lynx para detecção de alucinação
                prompt_lynx = f"Context: {contexto_referencia}\n\nStatement: {sent}\n\nIs the statement faithful to the context?"
                
                guard_resp = client.chat.completions.create(
                    model=MODEL_GUARD,
                    messages=[{"role": "user", "content": prompt_lynx}],
                    temperature=0.0
                )
                
                veredito = guard_resp.choices[0].message.content.lower()
                
                # Cálculo do Escore de Não-Conformidade (s_i)
                # s_i = 1 se alucinou, 0 se é fiel. 
                # (Na fase 3 usaremos logprobs para ter valores contínuos entre 0 e 1)
                si_score = 0.0 if "faithful" in veredito and "unfaithful" not in veredito else 1.0
                
                resultados_finais.append({
                    "id_bo": row.get('NUM_BO', idx),
                    "contexto_original": contexto_referencia,
                    "sentenca_gerada": sent,
                    "decisao_lynx": veredito,
                    "non_conformity_score": si_score
                })       
        except Exception as e:
            print(f"\nErro no processamento do registro {idx}: {e}")
            continue
    # Salvar resultados para o cálculo estatístico (Conformal Prediction)
    df_final = pd.DataFrame(resultados_finais)
    df_final.to_csv("dados_brutos_experimento.csv", index=False, encoding='utf-8')
    print(f"\nSucesso! Arquivo 'dados_brutos_experimento.csv' gerado.")        

In [61]:
# Execução
processar_experimento("Dataset/500-avaliacao_resumos.csv", 10)

print("Processamento concluído. Resultados armazenados em 'resultados_finais.json'.")

Dataset carregado com sucesso: 500 registros.
Iniciando auditoria de 10 registros policiais...


100%|██████████| 10/10 [5:56:52<00:00, 2141.23s/it] 


Sucesso! Arquivo 'dados_brutos_experimento.csv' gerado.
Processamento concluído. Resultados armazenados em 'resultados_finais.json'.
